# Note benchmark

Run All excludes only the exact Bach10/URMP recordings used in the parameter sweep and uses the promoted production defaults for Attune. Compatible Attune caches are reused; stale results are refreshed. All competitor methods are cache-only. Coco evaluation is unchanged.

The expanded evaluation retains 36 Bach10, 136 URMP, and 260 CocoChorales recordings (432 total). Other instrument stems from tuning pieces remain eligible.

Restart the kernel before Run All. Existing compatible per-recording results are reused across cohort changes; only missing or stale Attune recordings are computed. Competitor inference remains disabled. Summary tables and paired tests are rebuilt for the retained tracks. Historical piece-excluded outputs are preserved under `archive/`; cleared outputs must be regenerated before reporting numbers.


In [1]:
from dataclasses import replace
from pathlib import Path
import sys
REPO_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'app.py').is_file()))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from IPython.display import display
import pandas as pd
import importlib
for name in ("NoteDetectorBase", "NoteCache", "competitors.Ruptures", "competitors.SlopeWindow",
             "competitors.Attune", "competitors.BasicPitch", "competitors.CrepeNotes", "competitors.Tony",
             "competitors.MT3", "competitors.YourMT3",
             "NoteBenchmarker", "NoteNotebook"):
    importlib.reload(importlib.import_module(f"benchmarks.modules.note.{name}"))
from benchmarks.modules.note.NoteNotebook import NotebookConfig
from benchmarks.modules.note.NoteNotebook import NoteNotebook
from benchmarks.notebooks.scripts.evaluation_holdout import note_notebook, Holdout
from benchmarks.modules.pitch.PitchNotebook import PitchNotebook
PITCH_CONFIG = PitchNotebook.NotebookConfig()
# Production Attune plus the selected external competitors; no ruptures sweep.
CONFIG = NotebookConfig(methods=("attune", "basic-pitch", "crepe-notes", "tony",
                                 "mt3", "yourmt3"),
                        seed=0, workers=PITCH_CONFIG.workers, neural_workers=6,
                        force=False, force_methods=(), use_pitch_cache=True,
                        cache_only_methods=("basic-pitch", "crepe-notes", "tony", "mt3", "yourmt3"), max_tracks=None)
# None means all available stems; Coco uses the pitch paper's seeded sample.
SELECTION = {"coco": (20, 260), "urmp": (None, 149), "bach10-original": (None, 40)}
BENCHES = {dataset: note_notebook(replace(CONFIG, preliminary_tracks_per_instrument=count))
           for dataset, (count, expected) in SELECTION.items()}
BENCH = BENCHES["coco"]  # Common metric/reporting helpers.
BENCH.show_configuration()

from algorithms.Config import Config as PitchConfig
from algorithms.PitchSmoother import PitchSmoother
from benchmarks.modules.pitch.PitchCache import PitchCache
from benchmarks.modules.pitch.competitors.Attune import Attune as PitchAttune

selected_tracks = {}
for dataset, bench in BENCHES.items():
    selected_tracks[dataset] = bench.select_tracks(dataset)
display(pd.DataFrame([{'dataset': d, 'selected_tracks': len(t)} for d,t in selected_tracks.items()]))
print('Attune uses production sweep defaults; compatible pitch/note caches are reused.')
print('Competitor inference is disabled; missing competitor caches stop the run.')

# Preview compatible note checkpoints and pending note/transcription jobs.
for dataset, bench in BENCHES.items():
    display(bench.plan_preliminary(dataset).groupby(["dataset", "method", "action"])
            .size().rename("recordings").to_frame())

# Presentation labels only; retain stable metric keys in cached results.
METRIC_LABELS = {"f1": "Onset F1", "f1_offset": "Transcription F1"}
def display_metrics(frame):
    shown = frame.rename(columns=METRIC_LABELS).copy()
    if "metric" in shown.columns:
        shown["metric"] = shown["metric"].replace(METRIC_LABELS)
    for column in ("method", "competitor"):
        if column in shown.columns:
            shown[column] = shown[column].replace({"attune": "Attune"})
    display(shown)

held = Holdout()
held.output.mkdir(parents=True, exist_ok=True)
from benchmarks.notebooks.scripts.evaluation_holdout import write, sha
write(held.output / 'note_selection.json', dict(tuning_manifest=str(held.manifest),
      tuning_sha256=sha(held.manifest), policy='exact tuning-track exclusions',
      retained={d: [t['track_id'] for t in tracks] for d, tracks in selected_tracks.items()}))


,setting,value
0,workers,8
1,neural_workers,6
2,preliminary_tracks_per_instrument,20
3,seed,0
4,max_tracks,None
5,methods,"(attune, basic-pitch, crepe-notes, tony, mt3, ..."
6,onset_tolerance,0.05
7,pitch_tolerance,50.0
8,offset_ratio,0.2
9,offset_min_tolerance,0.05


,dataset,selected_tracks
0,coco,260
1,urmp,136
2,bach10-original,36


Attune uses production sweep defaults; compatible pitch/note caches are reused.
Competitor inference is disabled; missing competitor caches stop the run.


recordings
dataset method      action            
coco    attune      cached         260
        basic-pitch cached         260
        crepe-notes cached         260
        mt3         cached         260
        tony        cached         260
        yourmt3     cached         260

recordings
dataset method      action             
urmp    attune      cached          109
                    pending          27
        basic-pitch cached          136
        crepe-notes cached          136
        mt3         cached          136
        tony        cached          136
        yourmt3     cached          136

recordings
dataset         method      action             
bach10-original attune      cached           32
                            pending           4
                basic-pitch cached           36
                crepe-notes cached           36
                mt3         cached           36
                tony        cached           36
                yourmt3     cached           36

## Protocol and CPU timing

Seed 0 selects 20 stems per available Coco instrument (260 total), all 149 URMP stems, and all 40 original Bach10 stems using the existing pitch dataset samplers. Setup asserts these counts and unique track identities. Coco uses the test split, original isolated stem audio and stem MIDI. URMP uses isolated `AuSep` recordings and official `Notes` files: onset seconds, frequency Hz, duration seconds. Bach10 uses all 40 original recordings (ten each of violin, clarinet, saxophone and bassoon). Its `GTNotes.mat` references supply performed boundaries on the 10 ms grid: MATLAB frame 1 is centered at 23 ms, and first/last frame centers define onset/offset. Each note is assigned the median fractional MIDI pitch of its annotated contour, converted to Hz without rounding to a score pitch. Supplied score MIDI parts condition Attune separately; interpolated `.txt` score-alignment times are not used as note ground truth. Full recordings are scored. Selection manifests are saved before execution; missing selected inputs fail loudly.

Primary quality: **Transcription F1**, pooled-note (micro) F1 requiring **50 ms onset and 50 cents pitch** tolerance plus offsets within **max(50 ms, 20% of reference duration)**. **Onset F1** uses the same pitch/onset requirements with offsets ignored. Both metrics also report precision and recall. Empty predictions score zero; dependency/runtime failures remain errors, never zeros. Full polyphonic Basic Pitch predictions are retained, including simultaneous onsets.

Jobs run in fresh spawned processes, normally CPU count minus one workers. Heavy neural methods use up to six workers to bound memory. Every numerical runtime gets one thread; GPU execution is disabled. Results include:

| Column | Measured scope |
|---|---|
| `cpu_seconds` | Full algorithm CPU: frontend + note stage; a reused frontend retains its original measured CPU cost. Tony reports its complete call because stages are unavailable |
| `execution_cpu_seconds` | CPU actually spent in this invocation, including imports/setup/cache loading; a cache hit makes this smaller without making the algorithm appear free |
| `frontend_cpu_seconds` | Attune pitch detection+smoothing; Basic Pitch neural inference; CREPE tracking |
| `segmentation_cpu_seconds` | Attune score-span/minimum-length preparation + production KernelCPD; Basic Pitch native note creation; CREPE Notes native post-processing including amplitude analysis and MIDI writing |
| `wall_seconds` | Observed invocation elapsed time; cache- and contention-dependent diagnostic |
| `audio_per_cpu` | Total audio duration divided by total full-call CPU seconds |

CPU time is `process_time` plus reaped child user/system CPU, so Tony's Sonic Annotator work is counted. Tony exposes no separate segmentation timer: stage columns are missing, not zero. Stage boundaries are not identical algorithms: Basic Pitch has already inferred onset/note activations before note creation. Compare stage times with that qualification. CPU timing avoids charging scheduling waits, though contention can still affect CPU efficiency. Stage timings include different native loading/model-initialization costs: Attune follows the pitch notebook’s detector+smoother timing, while the external APIs load audio/models within their frontend. Neither aggregate CPU nor invocation time is steady-state interactive latency.

Attune reuses `pitch.ipynb`’s full smoothed `PitchData` cache, at the same sample rate, range, seed and selected stem IDs. Reuse requires the current cache version, production smoother version and process-CPU timing; the setup preflight stops on misses before launching evaluation. The underlying adapter can regenerate missing pitch data outside this notebook, so do not modify caches during a run. Old wall-clock or incompatible caches are not accepted. These legacy cache files do not contain full source/config fingerprints, so the compatibility guarantee is their version contract, not proof of arbitrary older settings; set `use_pitch_cache=False` to force a fresh frontend. Successful note checkpoints are fingerprinted and resumable. This notebook sets `cache_only_methods=()` so missing or incompatible note checkpoints are computed. The separate setup preflight requires compatible shared Attune pitch caches for all selected tracks. `force_methods=()` (the notebook default) resumes compatible jobs for every method. `force=True` deliberately bypasses reuse, and `force_methods` can explicitly rerun selected methods. Otherwise inputs, inference code, runtime dependencies, scoring settings, and saved success determine the schedule. Each run saves input/source hashes, versions, config, selection, predictions, per-job logs, raw rows, summaries and paired comparisons under `results/note/notebook_runs/`. CREPE Notes reuses the shared CREPE raw frequency/confidence cache (`crepe__<track>.raw.npz`) produced by the pitch evaluation when audio, model, preprocessing, dependency versions, and timing are compatible. Legacy thresholded CREPE estimates alone cannot supply the required confidence; missing or incompatible raw caches trigger CREPE inference and are saved for reuse. Basic Pitch’s pitch cache lacks note/onset activations, so it reuses complete predictions through compatible note-run checkpoints instead.

Live progress uses the exact `PitchBenchmarker.Progress` compact renderer from `pitch.ipynb`: one updating `[   n/total] method: track_id` line across all selected method/track pairs, with the same track IDs. Cached and newly computed tracks each advance this counter once; fully cached runs use the same display and reach the same total. Errors are listed after the live line finishes; full Python/native runtime output stays in each job’s log.


CREPE Notes retains its native CNN onset splitting. Its adapter resolves installed FFmpeg/ffprobe tools even when the notebook PATH omits Homebrew; this fixes the earlier decoder failures on CocoChorales and URMP. Missing decoder tools now fail before expensive pitch inference. Reports retain full provenance in separate directories. Completed jobs are reused across directories using per-recording input hashes, inference/scoring code, relevant dependencies, and scoring settings; worker counts, method selection, Git HEAD, and unrelated benchmark edits do not invalidate them. Compatible checkpoints from the stopped three-worker run are recognized automatically. Each returned job is atomically checkpointed before the status counter advances. Interruptions preserve completed jobs; with cache-only mode disabled, `force_methods=()` resumes only unfinished or failed jobs. `force=True` explicitly bypasses result reuse. CREPE raw frequency/confidence results are separately cached, so an interrupted note post-processing job can reuse its completed CREPE frontend.


Attune now includes production score alignment, robust timing refitting, and repeat-note recovery on every dataset. Recovered notes are scored against the unchanged references. `refinement_cpu_seconds` records this additional stage and is included in `cpu_seconds`. The optional `attune-audio-only` diagnostic remains score-free and excludes recovery.


## Expanded CocoChorales

This sample may overlap earlier note sweeps; it is a development check. Do not treat the test-split name as proof it was never used for model selection.

In [2]:
expanded_coco = BENCHES["coco"].run_preliminary("coco")
display_metrics(BENCH.summarize(expanded_coco))


Reusing completed run: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/note/notebook_runs/preliminary_coco_945b13ab9d698ebf
[1560/1560] yourmt3: test__woodwind_track229974__4_bassoon      
Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/note/notebook_runs/preliminary_coco_945b13ab9d698ebf


,dataset,method,attempted,succeeded,failed,datasets_scored,complete,aggregation,tp,fp,...,precision_offset,recall_offset,Transcription F1,cpu_seconds,wall_seconds,execution_cpu_seconds,frontend_cpu_seconds,segmentation_cpu_seconds,refinement_cpu_seconds,audio_per_cpu
0,coco,Attune,260,260,0,1,True,pooled_note_counts,5881,297,...,0.946908,0.947982,0.947445,2.775917,0.240391,0.229279,2.727964,0.033424,0.01453,7.712970
1,coco,basic-pitch,260,260,0,1,True,pooled_note_counts,5848,3925,...,0.509158,0.806352,0.624185,5.301005,8.008404,7.596056,5.285692,0.015313,NaN,4.038964
2,coco,crepe-notes,260,260,0,1,True,pooled_note_counts,5969,766,...,0.875575,0.955599,0.913839,111.535432,3.735067,3.636211,110.758412,0.777020,NaN,0.191962
3,coco,mt3,260,260,0,1,True,pooled_note_counts,5670,2823,...,0.597904,0.822881,0.692580,14.397003,11.259992,14.397003,NaN,NaN,NaN,1.487155
4,coco,tony,260,260,0,1,True,pooled_note_counts,3673,1962,...,0.611535,0.558418,0.583771,1.394020,1.258300,1.181475,NaN,NaN,NaN,15.358869
5,coco,yourmt3,260,260,0,1,True,pooled_note_counts,5328,556,...,0.796057,0.759034,0.777105,19.561178,20.194717,19.561178,NaN,NaN,NaN,1.094544


## Expanded URMP

Current production pitch settings were selected using the two-per-instrument URMP development sample. These results are not independent held-out evidence. Attune's score-informed row and the audio-only external rows have different input privileges.

In [3]:
expanded_urmp = BENCHES["urmp"].run_preliminary("urmp")
display_metrics(BENCH.summarize(expanded_urmp))


Running: preliminary_urmp (789 cached, 27 pending)
[ 816/816] yourmt3: 44_K515_vn_vn_va_va_vc/AuSep_5_vc_44_K515                  
Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/note/notebook_runs/preliminary_urmp_a44e717f96dc867e


,dataset,method,attempted,succeeded,failed,datasets_scored,complete,aggregation,tp,fp,...,precision_offset,recall_offset,Transcription F1,cpu_seconds,wall_seconds,execution_cpu_seconds,frontend_cpu_seconds,segmentation_cpu_seconds,refinement_cpu_seconds,audio_per_cpu
0,urmp,Attune,136,136,0,1,True,pooled_note_counts,23848,2756,...,0.862276,0.853042,0.857634,18.170203,2.648288,2.561647,16.750328,0.123753,1.296122,6.161987
1,urmp,basic-pitch,136,136,0,1,True,pooled_note_counts,23059,8875,...,0.581387,0.690391,0.631217,8.670905,9.632419,9.500761,8.504475,0.166430,NaN,12.912671
2,urmp,crepe-notes,136,136,0,1,True,pooled_note_counts,23762,7658,...,0.680108,0.794623,0.732919,581.364112,20.422796,20.112215,577.439729,3.924383,NaN,0.192589
3,urmp,mt3,136,136,0,1,True,pooled_note_counts,25310,5508,...,0.762217,0.873494,0.814070,40.144189,25.001667,40.144189,NaN,NaN,NaN,2.789060
4,urmp,tony,136,136,0,1,True,pooled_note_counts,12550,2060,...,0.774127,0.420571,0.545034,3.584393,2.957500,2.887539,NaN,NaN,NaN,31.236687
5,urmp,yourmt3,136,136,0,1,True,pooled_note_counts,24508,1740,...,0.863647,0.842964,0.853180,119.294795,121.930981,119.294795,NaN,NaN,NaN,0.938554


## Expanded Bach10 — original recordings

The balanced selection matches `pitch.ipynb`, allowing Attune to reuse compatible production pitch caches. The four instrument parts are evaluated separately; ensemble mixes are excluded. Frame references were generated with YIN and manual corrections, then grouped into notes manually. See the [release description](https://labsites.rochester.edu/air/datasets/Bach10%20Dataset_v1.0.pdf).

This uses the same spawned worker pool, neural worker limit, single updating status line, metrics, and report format as the CocoChorales and URMP cells. The default run has 320 rows (40 recordings × 8 methods), including cached baselines. Results and manifests are saved under a separate `preliminary_bach10-original_*` directory. The sample is a development comparison, not independent held-out evidence.


In [4]:
expanded_bach10 = BENCHES["bach10-original"].run_preliminary("bach10-original")
display_metrics(BENCH.summarize(expanded_bach10))


Running: preliminary_bach10-original (212 cached, 4 pending)
[ 216/216] yourmt3: 10-NunBitten-bassoon                       
Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/note/notebook_runs/preliminary_bach10-original_1a7b817e17eccdf1


,dataset,method,attempted,succeeded,failed,datasets_scored,complete,aggregation,tp,fp,...,precision_offset,recall_offset,Transcription F1,cpu_seconds,wall_seconds,execution_cpu_seconds,frontend_cpu_seconds,segmentation_cpu_seconds,refinement_cpu_seconds,audio_per_cpu
0,bach10-original,Attune,36,36,0,1,True,pooled_note_counts,1524,200,...,0.829466,0.803371,0.816210,4.844144,0.735158,0.714168,4.431613,0.041453,0.371078,6.824964
1,bach10-original,basic-pitch,36,36,0,1,True,pooled_note_counts,1619,625,...,0.668004,0.842135,0.745030,5.479982,6.569591,6.382023,5.456404,0.023578,NaN,6.033069
2,bach10-original,crepe-notes,36,36,0,1,True,pooled_note_counts,1614,300,...,0.780042,0.838764,0.808338,173.169893,135.568385,134.768926,172.109389,1.060504,NaN,0.190917
3,bach10-original,mt3,36,36,0,1,True,pooled_note_counts,1481,351,...,0.718341,0.739326,0.728682,17.003717,12.369669,17.003717,NaN,NaN,NaN,1.944346
4,bach10-original,tony,36,36,0,1,True,pooled_note_counts,1580,139,...,0.865038,0.835393,0.849957,1.972577,1.448426,1.395731,NaN,NaN,NaN,16.760363
5,bach10-original,yourmt3,36,36,0,1,True,pooled_note_counts,1419,233,...,0.777845,0.721910,0.748834,31.336293,31.819224,31.336293,NaN,NaN,NaN,1.055042


## Paired quality, coverage and instrument breakdown

Differences are pooled-note micro-F1 for Attune minus baseline on successful matching instrument tracks, reported separately per corpus. Exploratory 95% percentile intervals resample whole pieces/tracks (2,000 draws), keeping sibling stems together. They do not establish equivalence: a “comparable” claim needs a prespecified non-inferiority margin and independent evaluation. `complete=False` flags missing pairs. Do not rank methods using unequal coverage without reporting it.

In [5]:
rows = pd.concat(
    [expanded_coco, expanded_urmp, expanded_bach10], ignore_index=True
)
display_metrics(BENCH.summarize(rows))
display_metrics(BENCH.paired_comparisons(rows))
display_metrics(rows.loc[rows.status != "ok", ["dataset", "method", "track_id", "error"]])
good = rows[rows.status == "ok"]
if not good.empty:
    display_metrics(
        pd.concat(
            [
                BENCH.summarize(group).assign(instrument=instrument)
                for (_, instrument), group in good.groupby(["dataset", "instrument"])
            ],
            ignore_index=True,
        )[
            [
                "dataset",
                "instrument",
                "method",
                "succeeded",
                "tp",
                "fp",
                "fn",
                "f1",
                "f1_offset",
            ]
        ]
    )


,dataset,method,attempted,succeeded,failed,datasets_scored,complete,aggregation,tp,fp,...,precision_offset,recall_offset,Transcription F1,cpu_seconds,wall_seconds,execution_cpu_seconds,frontend_cpu_seconds,segmentation_cpu_seconds,refinement_cpu_seconds,audio_per_cpu
0,bach10-original,Attune,36,36,0,1,True,pooled_note_counts,1524,200,...,0.829466,0.803371,0.816210,4.844144,0.735158,0.714168,4.431613,0.041453,0.371078,6.824964
1,bach10-original,basic-pitch,36,36,0,1,True,pooled_note_counts,1619,625,...,0.668004,0.842135,0.745030,5.479982,6.569591,6.382023,5.456404,0.023578,NaN,6.033069
2,bach10-original,crepe-notes,36,36,0,1,True,pooled_note_counts,1614,300,...,0.780042,0.838764,0.808338,173.169893,135.568385,134.768926,172.109389,1.060504,NaN,0.190917
3,bach10-original,mt3,36,36,0,1,True,pooled_note_counts,1481,351,...,0.718341,0.739326,0.728682,17.003717,12.369669,17.003717,NaN,NaN,NaN,1.944346
4,bach10-original,tony,36,36,0,1,True,pooled_note_counts,1580,139,...,0.865038,0.835393,0.849957,1.972577,1.448426,1.395731,NaN,NaN,NaN,16.760363
5,bach10-original,yourmt3,36,36,0,1,True,pooled_note_counts,1419,233,...,0.777845,0.721910,0.748834,31.336293,31.819224,31.336293,NaN,NaN,NaN,1.055042
6,coco,Attune,260,260,0,1,True,pooled_note_counts,5881,297,...,0.946908,0.947982,0.947445,2.775917,0.240391,0.229279,2.727964,0.033424,0.014530,7.712970
7,coco,basic-pitch,260,260,0,1,True,pooled_note_counts,5848,3925,...,0.509158,0.806352,0.624185,5.301005,8.008404,7.596056,5.285692,0.015313,NaN,4.038964
8,coco,crepe-notes,260,260,0,1,True,pooled_note_counts,5969,766,...,0.875575,0.955599,0.913839,111.535432,3.735067,3.636211,110.758412,0.777020,NaN,0.191962
9,coco,mt3,260,260,0,1,True,pooled_note_counts,5670,2823,...,0.597904,0.822881,0.692580,14.397003,11.259992,14.397003,NaN,NaN,NaN,1.487155


,dataset,baseline,metric,aggregation,paired_tracks,groups,delta,ci_low,ci_high,complete
0,bach10-original,basic-pitch,Onset F1,pooled_note_counts,36,10,0.065191,0.039201,0.092001,True
1,bach10-original,crepe-notes,Onset F1,pooled_note_counts,36,10,-0.003986,-0.033097,0.025277,True
2,bach10-original,tony,Onset F1,pooled_note_counts,36,10,-0.033252,-0.055336,-0.012228,True
3,bach10-original,mt3,Onset F1,pooled_note_counts,36,10,0.049819,0.027675,0.072054,True
4,bach10-original,yourmt3,Onset F1,pooled_note_counts,36,10,0.042940,0.024949,0.062281,True
5,bach10-original,basic-pitch,Transcription F1,pooled_note_counts,36,10,0.071180,0.033254,0.108649,True
6,bach10-original,crepe-notes,Transcription F1,pooled_note_counts,36,10,0.007872,-0.046355,0.057862,True
7,bach10-original,tony,Transcription F1,pooled_note_counts,36,10,-0.033747,-0.066778,-0.000228,True
8,bach10-original,mt3,Transcription F1,pooled_note_counts,36,10,0.087528,0.044646,0.126914,True
9,bach10-original,yourmt3,Transcription F1,pooled_note_counts,36,10,0.067376,0.034222,0.100161,True


,dataset,method,track_id,error


,dataset,instrument,method,succeeded,tp,fp,fn,Onset F1,Transcription F1
0,bach10-original,bassoon,Attune,9,454,27,31,0.939959,0.917184
1,bach10-original,bassoon,basic-pitch,9,444,316,41,0.713253,0.689157
2,bach10-original,bassoon,crepe-notes,9,459,77,26,0.899119,0.852106
3,bach10-original,bassoon,mt3,9,443,69,42,0.888666,0.760281
4,bach10-original,bassoon,tony,9,460,24,25,0.949432,0.928793
...,...,...,...,...,...,...,...,...,...
175,urmp,violin,basic-pitch,33,6558,1782,837,0.833556,0.626120
176,urmp,violin,crepe-notes,33,6797,1451,598,0.869015,0.784249
177,urmp,violin,mt3,33,6984,1185,411,0.897456,0.833205
178,urmp,violin,tony,33,2354,496,5041,0.459541,0.391410


## Competitor follow-up and claim boundary

The historical results include Basic Pitch, CREPE Notes and Tony plus PELT (L1/L2/RBF), KernelCPD (linear/Gaussian/cosine), bottom-up and window segmentation; a one-track neural-boundary experiment is not a meaningful competitor benchmark. Current configuration also lists oracle-count Dynp and slope-aware windows. Official JAX MT3 (`mt3`) is included by default; `mt3-port.ipynb` provides setup and bounded validation.

Further candidates: [YourMT3+](https://arxiv.org/abs/2407.04822) for stronger multi-instrument transcription, plus the newly released [MuScriptor (2026)](https://arxiv.org/abs/2607.08168); audit checkpoint training overlap and use appropriate URMP held-out splits before evaluating them. [Melodia → notes](https://github.com/justinsalamon/audio_to_midi_melodia) is a useful classical sanity baseline, but its authors describe the segmenter as deliberately simple. Tony is already represented and should remain in the default experiment. Piano-only transcription systems are not direct replacements for this varied-instrument suite.

The historical descriptive result is: “In the saved CocoChorales benchmark, the linear KernelCPD configuration achieved higher mean onset-and-pitch F1 than Basic Pitch and CREPE Notes on the same 2,486 stems.” A present-tense claim about current Attune requires the fresh results, explicit input/metric conditions, and preferably an untouched evaluation set. Neither notebook establishes performance on mixtures, live microphone recordings, or unseen instruments.

## Overall metrics across datasets

Pool true positives, false positives, and false negatives across all successful instrument tracks in all three datasets, then compute micro precision, recall, and F1. Matching is performed separately within each recording. **Onset F1** requires matching pitch and onset (50 cents / 50 ms), ignoring offsets. **Transcription F1** additionally requires the offset within max(50 ms, 20% of reference duration). Both are pooled micro-F1; the overall table ranks by Transcription F1. Failed recordings remain reported as failures, not zero scores; `complete` flags incomplete coverage. CPU time remains an average per successful recording, and `audio_per_cpu` divides total audio duration by total algorithm CPU time.


In [6]:
overall_rows = pd.concat(
    [expanded_coco, expanded_urmp, expanded_bach10], ignore_index=True
)
# Require complete coverage of every retained evaluation track.
expected_ids = {(dataset, track["track_id"]) for dataset, tracks in selected_tracks.items() for track in tracks}
for method in CONFIG.methods:
    method_rows = overall_rows.loc[overall_rows.method == method]
    actual_ids = set(zip(method_rows.dataset, method_rows.track_id))
    if len(method_rows) != len(expected_ids) or actual_ids != expected_ids or not method_rows.status.eq("ok").all():
        raise RuntimeError(f"{method}: incomplete expanded evaluation; inspect the failure report and rerun to resume.")
overall_summary = BENCH.summarize(overall_rows, by_dataset=False).set_index("method")
quality_metrics = [
    "precision",
    "recall",
    "f1",
    "precision_offset",
    "recall_offset",
    "f1_offset",
]
overall_summary[quality_metrics] *= 100
overall_summary = (
    overall_summary[
        [
            "attempted",
            "succeeded",
            "failed",
            "datasets_scored",
            "complete",
            "tp",
            "fp",
            "fn",
            *quality_metrics[:3],
            "tp_offset",
            "fp_offset",
            "fn_offset",
            *quality_metrics[3:],
            "cpu_seconds",
            "audio_per_cpu",
        ]
    ]
    .rename(
        columns={
            "precision": "Onset precision (%)",
            "recall": "Onset recall (%)",
            "f1": "Onset F1 (%)",
            "precision_offset": "Transcription precision (%)",
            "recall_offset": "Transcription recall (%)",
            "f1_offset": "Transcription F1 (%)",
            "cpu_seconds": "CPU seconds / recording",
        }
    )
    .sort_values("Transcription F1 (%)", ascending=False)
)
display(
    overall_summary.rename(index={"attune": "Attune"}).style.format(
        {
            column: "{:.2f}"
            for column in overall_summary.columns
            if "(%)" in column or column in ("CPU seconds / recording", "audio_per_cpu")
        },
        na_rep="—",
    )
)


,attempted,succeeded,failed,datasets_scored,complete,tp,fp,fn,Onset precision (%),Onset recall (%),Onset F1 (%),tp_offset,fp_offset,fn_offset,Transcription precision (%),Transcription recall (%),Transcription F1 (%),CPU seconds / recording,audio_per_cpu
method,,,,,,,,,,,,,,,,,,,
Attune,432,432,0,3,True,31253,3253,3590,90.57,89.70,90.13,30220,4286,4623,87.58,86.73,87.15,7.79,6.53
yourmt3,432,432,0,3,True,31255,2529,3588,92.51,89.70,91.09,28638,5146,6205,84.77,82.19,83.46,51.94,0.98
mt3,432,432,0,3,True,32461,8682,2382,78.90,93.16,85.44,29884,11259,4959,72.63,85.77,78.66,22.72,2.24
crepe-notes,432,432,0,3,True,31345,8724,3498,78.23,89.96,83.68,28759,11310,6084,71.77,82.54,76.78,264.58,0.19
basic-pitch,432,432,0,3,True,30526,13425,4317,69.45,87.61,77.48,25041,18910,9802,56.97,71.87,63.56,6.38,7.98
tony,432,432,0,3,True,17803,4161,17040,81.06,51.09,62.68,16243,5721,18600,73.95,46.62,57.19,2.13,23.87


## Paired significance of overall Transcription F1

Recompute two-sided paired permutation tests from `overall_rows` above, without rerunning inference. The default family is Attune versus Basic Pitch, CREPE Notes, Tony, MT3, and YourMT3+ on **Transcription F1 (pitch + onset + offset micro-F1)**, with Holm correction across the five comparisons at 0.05. Positive differences favor Attune. Adding `"f1"` to `SIGNIFICANCE_METRICS` includes those comparisons in the same Holm family.

All comparisons use the common successful track intersection. Swaps keep whole source groups together, conservatively merging repeated URMP titles, and recompute pooled F1 from swapped TP/FP/FN counts. Small designs enumerate every swap; larger designs use 9,999 random swaps with a plus-one correction. Coverage reports excluded tracks; failed runs are never assigned zero scores. `SOURCE_GROUPS` can link known shared compositions across datasets. The same cell supports larger runs by consuming their rows from the preceding overall-results cell; significance does not establish held-out generalization or superiority for every instrument.


In [7]:
import benchmarks.modules.note.NoteBenchmarker as note_significance_module
importlib.reload(note_significance_module)
SIGNIFICANCE_METHODS = CONFIG.methods
SIGNIFICANCE_METRICS = ('f1_offset',)
SIGNIFICANCE_RESAMPLES = 9999
SIGNIFICANCE_ALPHA = 0.05
SOURCE_GROUPS = {}
note_significance, significance_coverage = note_significance_module.NoteEvaluation.paired_significance(overall_rows, methods=SIGNIFICANCE_METHODS, metrics=SIGNIFICANCE_METRICS, n_resamples=SIGNIFICANCE_RESAMPLES, seed=CONFIG.seed, alpha=SIGNIFICANCE_ALPHA, source_groups=SOURCE_GROUPS)
display(significance_coverage)
significance_display = note_significance.rename(columns={
    'attune_f1': 'Attune Transcription F1 (%)',
    'competitor_f1': 'Competitor Transcription F1 (%)',
}).replace({'metric': METRIC_LABELS})
display(significance_display.style.format({'Attune Transcription F1 (%)': '{:.2f}', 'Competitor Transcription F1 (%)': '{:.2f}', 'difference_pp': '{:+.2f}', 'p_value': '{:.4g}', 'p_adj': '{:.4g}', 'alpha': '{:.2f}'}))
print('Transcription F1 columns are percentages; difference_pp is Attune minus competitor.')
print('significant tests p_adj <= alpha (two-sided); inspect difference_pp for direction.')


,dataset,method,selected_tracks,attempted,succeeded,paired_tracks,excluded_tracks
0,bach10-original,attune,36,36,36,36,0
1,bach10-original,basic-pitch,36,36,36,36,0
2,bach10-original,crepe-notes,36,36,36,36,0
3,bach10-original,tony,36,36,36,36,0
4,bach10-original,mt3,36,36,36,36,0
5,bach10-original,yourmt3,36,36,36,36,0
6,coco,attune,260,260,260,260,0
7,coco,basic-pitch,260,260,260,260,0
8,coco,crepe-notes,260,260,260,260,0
9,coco,tony,260,260,260,260,0


,method,metric,tracks,source_groups,excluded_tracks,Attune Transcription F1 (%),Competitor Transcription F1 (%),difference_pp,p_value,exact,permutations,seed,p_adj,significant,alpha
0,basic-pitch,Transcription F1,432,293,0,87.15,63.56,+23.59,0.0001,False,9999,0,0.0005,True,0.05
1,crepe-notes,Transcription F1,432,293,0,87.15,76.78,+10.37,0.0001,False,9999,0,0.0005,True,0.05
2,tony,Transcription F1,432,293,0,87.15,57.19,+29.97,0.0001,False,9999,0,0.0005,True,0.05
3,mt3,Transcription F1,432,293,0,87.15,78.66,+8.50,0.0001,False,9999,0,0.0005,True,0.05
4,yourmt3,Transcription F1,432,293,0,87.15,83.46,+3.69,0.0001,False,9999,0,0.0005,True,0.05


Transcription F1 columns are percentages; difference_pp is Attune minus competitor.
significant tests p_adj <= alpha (two-sided); inspect difference_pp for direction.


## Paired significance of overall Onset F1

Run after the Transcription F1 significance cell above. Compare Attune with each competitor on **Onset F1 (pitch + onset micro-F1, ignoring offsets)** using the same overall rows, source groups, seed, and permutation settings, without rerunning inference. Holm correction is applied separately across the five onset comparisons at 0.05; it does not jointly correct onset and transcription comparisons. To test both metrics as one family, use `SIGNIFICANCE_METRICS = ('f1_offset', 'f1')` in the preceding cell.

Positive `difference_pp` favors Attune; negative values favor the competitor. The `significant` column indicates whether the two-sided Holm-adjusted p-value is at or below alpha. Coverage reports the common successful track intersection and excluded tracks.


In [8]:
import benchmarks.modules.note.NoteBenchmarker as note_significance_module
importlib.reload(note_significance_module)
ONSET_SIGNIFICANCE_METRICS = ('f1',)
onset_significance, onset_significance_coverage = note_significance_module.NoteEvaluation.paired_significance(overall_rows, methods=SIGNIFICANCE_METHODS, metrics=ONSET_SIGNIFICANCE_METRICS, n_resamples=SIGNIFICANCE_RESAMPLES, seed=CONFIG.seed, alpha=SIGNIFICANCE_ALPHA, source_groups=SOURCE_GROUPS)
display(onset_significance_coverage)
onset_significance_display = onset_significance.rename(columns={
    'attune_f1': 'Attune Onset F1 (%)',
    'competitor_f1': 'Competitor Onset F1 (%)',
}).replace({'metric': METRIC_LABELS})
display(onset_significance_display.style.format({'Attune Onset F1 (%)': '{:.2f}', 'Competitor Onset F1 (%)': '{:.2f}', 'difference_pp': '{:+.2f}', 'p_value': '{:.4g}', 'p_adj': '{:.4g}', 'alpha': '{:.2f}'}))
print('Onset F1 columns are percentages; difference_pp is Attune minus competitor.')
print('significant tests p_adj <= alpha (two-sided); inspect difference_pp for direction.')


,dataset,method,selected_tracks,attempted,succeeded,paired_tracks,excluded_tracks
0,bach10-original,attune,36,36,36,36,0
1,bach10-original,basic-pitch,36,36,36,36,0
2,bach10-original,crepe-notes,36,36,36,36,0
3,bach10-original,tony,36,36,36,36,0
4,bach10-original,mt3,36,36,36,36,0
5,bach10-original,yourmt3,36,36,36,36,0
6,coco,attune,260,260,260,260,0
7,coco,basic-pitch,260,260,260,260,0
8,coco,crepe-notes,260,260,260,260,0
9,coco,tony,260,260,260,260,0


,method,metric,tracks,source_groups,excluded_tracks,Attune Onset F1 (%),Competitor Onset F1 (%),difference_pp,p_value,exact,permutations,seed,p_adj,significant,alpha
0,basic-pitch,Onset F1,432,293,0,90.13,77.48,+12.65,0.0001,False,9999,0,0.0005,True,0.05
1,crepe-notes,Onset F1,432,293,0,90.13,83.68,+6.45,0.0001,False,9999,0,0.0005,True,0.05
2,tony,Onset F1,432,293,0,90.13,62.68,+27.45,0.0001,False,9999,0,0.0005,True,0.05
3,mt3,Onset F1,432,293,0,90.13,85.44,+4.69,0.0002,False,9999,0,0.0005,True,0.05
4,yourmt3,Onset F1,432,293,0,90.13,91.09,-0.95,0.4795,False,9999,0,0.4795,False,0.05


Onset F1 columns are percentages; difference_pp is Attune minus competitor.
significant tests p_adj <= alpha (two-sided); inspect difference_pp for direction.
